# 42 — SC bit-flip sensitivity

High-fidelity **Perfect Integrator** (polyglot-complete) provides a clean
spike counter under constant current. The stochastic-computing section uses
the production `FaultInjector.inject_bit_flips` path on both unipolar operands.

## Honesty box

| | |
|---|---|
| **Proves** | Seeded production fault injection changes SC unipolar multiplication error across controlled BER values; Perfect Integrator spikes under clean constant current. |
| **Does not prove** | Superiority over fixed-point arithmetic, radiation hardness, automotive FIT rates, or robustness for every workload. |
| **Artefacts** | Local figures only. |
| **Models** | `PerfectIntegratorNeuron` (polyglot-complete). |


In [ ]:
from __future__ import annotations

import matplotlib.pyplot as plt
import numpy as np

from sc_neurocore import BitstreamEncoder, bitstream_to_probability
from sc_neurocore.neurons.models import PerfectIntegratorNeuron
from sc_neurocore.utils.fault_injection import FaultInjector

np.random.seed(7)
print("SC-NeuroCore — NB-42 fault-tolerance theatre")


## 1. Clean Perfect Integrator baseline


In [ ]:
neuron = PerfectIntegratorNeuron()
v, spikes = neuron.simulate(1000, current=2.0)
print(f"PerfectIntegrator: spikes={spikes}, vmax={float(v.max()):.3f}, dt={neuron.dt}")
t = np.arange(len(v)) * float(neuron.dt)
fig, ax = plt.subplots(figsize=(8, 2.5))
ax.plot(t, v, lw=0.9)
ax.set_title(f"Perfect Integrator sawtooth — spikes={spikes}")
ax.set_xlabel("time")
ax.set_ylabel("v")
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()


## 2. Production SC bit-flip injection


In [ ]:
def sc_product_with_faults(
    a: float, b: float, length: int, ber: float, seed: int
) -> float:
    ea = BitstreamEncoder(0.0, 1.0, length=length, seed=seed)
    eb = BitstreamEncoder(0.0, 1.0, length=length, seed=seed + 17)
    ba = np.asarray(ea.encode(a), dtype=np.uint8)
    bb = np.asarray(eb.encode(b), dtype=np.uint8)
    np.random.seed(seed + 101)
    ba = FaultInjector.inject_bit_flips(ba, ber)
    np.random.seed(seed + 202)
    bb = FaultInjector.inject_bit_flips(bb, ber)
    return float(bitstream_to_probability(ba & bb))

a, b = 0.7, 0.55
true_product = a * b
length = 2048
bers = np.array([0.0, 0.01, 0.02, 0.05, 0.08, 0.1, 0.15, 0.2])
trials = 24
sc_errs = []
for ber in bers:
    estimates = [
        sc_product_with_faults(a, b, length, float(ber), seed=1000 + trial)
        for trial in range(trials)
    ]
    sc_errs.append(float(np.mean(np.abs(np.asarray(estimates) - true_product))))

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(bers, sc_errs, "o-", label="production-injected SC error")
ax.set_xlabel("bit error rate (BER)")
ax.set_ylabel("mean |estimate − clean product|")
ax.set_title(
    f"Unipolar product a={a}, b={b}, clean={true_product:.3f}, L={length}, trials={trials}"
)
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()
plt.show()
print("mean SC errors", list(zip(bers.tolist(), np.round(sc_errs, 5))))
print("NB-42 complete: production FaultInjector path, no fixed-point comparison claim.")
